In [4]:
import json
import os
import re
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

CLUSTERS_FOLDER = Path('/Users/narikenzhe/Desktop/My_Jupyter/кластера 2026')
print("Start")


Start


In [5]:
def extract_js_literal(text: str, variable_name: str):
    marker = f"const {variable_name} ="
    start_idx = text.find(marker)
    if start_idx == -1: return None
    
    idx = start_idx + len(marker)
    while idx < len(text) and text[idx].isspace(): idx += 1
    if idx >= len(text): return None
    
    opening = text[idx]
    closing = '}' if opening == '{' else ']'
    depth = 0
    in_string, escaped, quote_char = False, False, ''
    
    for end_idx in range(idx, len(text)):
        char = text[end_idx]
        if in_string:
            if escaped: escaped = False
            elif char == '\\': escaped = True
            elif char == quote_char: in_string = False
            continue
        if char in ('\"', "'"): in_string, quote_char = True, char
        elif char == opening: depth += 1
        elif char == closing:
            depth -= 1
            if depth == 0:
                try: return json.loads(text[idx:end_idx + 1])
                except: return None
    return None

def get_city_from_filename(filename: str):
    city = filename.replace('_cluster_map.html', '')
    return city


In [8]:
def load_clusters_data(folder_path):
    all_polygons = []
    all_points = []
    
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Папка не найдена: {folder}")
        
    for filename in sorted(os.listdir(folder)):
        if not filename.endswith('.html'): continue
        
        city = get_city_from_filename(filename)
        file_path = folder / filename
        
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
            # 1. Извлекаем Полигоны (df_clusters)
            poly_data = extract_js_literal(content, 'clustersData')
            if poly_data:
                for feature in poly_data.get('features', []):
                    props = feature.get('properties', {}).copy()
                    if feature.get('geometry'):
                        props['geometry'] = shape(feature['geometry'])
                        props['city'] = city
                        props['source_file'] = filename
                        all_polygons.append(props)
            
            point_data = extract_js_literal(content, 'pointsData')
            if point_data:
                for feature in point_data.get('features', []):
                    props = feature.get('properties', {}).copy()
                    coords = feature.get('geometry', {}).get('coordinates', [])
                    if len(coords) == 2:
                        props['lon'], props['lat'] = coords[0], coords[1]
                        props['city'] = city
                        props['source_file'] = filename
                        all_points.append(props)
                        
    gdf_clusters = gpd.GeoDataFrame(all_polygons, geometry='geometry', crs='EPSG:4326') if all_polygons else gpd.GeoDataFrame()
    df_points = pd.DataFrame(all_points) if all_points else pd.DataFrame()
    
    return gdf_clusters, df_points

print("Начинаю загрузку данных из папки кластеров...")
df_clusters, df_points = load_clusters_data(CLUSTERS_FOLDER)
print(f"Загружено полигонов: {len(df_clusters)}")
print(f"Загружено точек: {len(df_points)}")

print("\nДанные доступны в переменных df_points.")


Начинаю загрузку данных из папки кластеров...
Загружено полигонов: 248
Загружено точек: 222678

Данные доступны в переменных df_points.


In [9]:
df_points.head()

,address,примечание,наличие услуг,cluster,service_type,color,lon,lat,city,source_file
0,"Г.АКТАУ,13,25,Б",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.140469,43.663839,aktau,aktau_cluster_map.html
1,"Г.АКТАУ,14,18,А",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.146722,43.650292,aktau,aktau_cluster_map.html
2,"Г.АКТАУ,14,26",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.146407,43.651682,aktau,aktau_cluster_map.html
3,"Г.АКТАУ,14,35,*",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.144063,43.654108,aktau,aktau_cluster_map.html
4,"Г.АКТАУ,14,57",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.144125,43.658602,aktau,aktau_cluster_map.html


In [12]:
WORK_DIR = Path("/Users/narikenzhe/Desktop/My_Jupyter/Work_Files")

In [13]:
Main_ADSL = WORK_DIR / "adsl_yandex_geo_3.csv"

In [15]:
main_df = pd.read_csv(Main_ADSL, sep=';', encoding='cp1251')

In [16]:
main_df.head()

,server_id,customer_account_id,town_state_id,town_state,adr_house,swap_segm,point_lat,point_lon
0,48.0,7166405.0,5792.0,г.Караганда,"г.Караганда,ул.ЭНГЕЛЬСА, д.4",1,49.883254,73.194693
1,114.0,6127603.0,11860.0,г.Семей,"г.Семей,ул.М?ратхан Бейсембаев, д.9",1,50.402369,80.204688
2,89.0,7930994.0,797.0,г.Актобе,"г.Актобе,ул.Братьев Жубановых, д.42",1,50.314685,57.132439
3,90.0,671046.0,2.0,г.Алматы,"г.Алматы,мкр.Нуршашкан, ул. Орен, д.30",1,43.336092,77.016684
4,64.0,6479898.0,3302.0,г.Уральск,"г.Уральск,ул.Алматы, д.42",1,51.230761,51.415102


In [17]:
merged_df = main_df.join(
    df_points,
    how = "inner"
)

In [23]:
merged_df.head()

,server_id,customer_account_id,town_state_id,town_state,adr_house,swap_segm,point_lat,point_lon,address,примечание,наличие услуг,cluster,service_type,color,lon,lat,city,source_file
0,48.0,7166405.0,5792.0,г.Караганда,"г.Караганда,ул.ЭНГЕЛЬСА, д.4",1,49.883254,73.194693,"Г.АКТАУ,13,25,Б",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.140469,43.663839,aktau,aktau_cluster_map.html
1,114.0,6127603.0,11860.0,г.Семей,"г.Семей,ул.М?ратхан Бейсембаев, д.9",1,50.402369,80.204688,"Г.АКТАУ,14,18,А",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.146722,43.650292,aktau,aktau_cluster_map.html
2,89.0,7930994.0,797.0,г.Актобе,"г.Актобе,ул.Братьев Жубановых, д.42",1,50.314685,57.132439,"Г.АКТАУ,14,26",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.146407,43.651682,aktau,aktau_cluster_map.html
3,90.0,671046.0,2.0,г.Алматы,"г.Алматы,мкр.Нуршашкан, ул. Орен, д.30",1,43.336092,77.016684,"Г.АКТАУ,14,35,*",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.144063,43.654108,aktau,aktau_cluster_map.html
4,64.0,6479898.0,3302.0,г.Уральск,"г.Уральск,ул.Алматы, д.42",1,51.230761,51.415102,"Г.АКТАУ,14,57",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.144125,43.658602,aktau,aktau_cluster_map.html


In [24]:
print(len(merged_df))

69231


In [26]:
merged_df.drop_duplicates(subset=["customer_account_id"])

,server_id,customer_account_id,town_state_id,town_state,adr_house,swap_segm,point_lat,point_lon,address,примечание,наличие услуг,cluster,service_type,color,lon,lat,city,source_file
0,48.0,7166405.0,5792.0,г.Караганда,"г.Караганда,ул.ЭНГЕЛЬСА, д.4",1,49.883254,73.194693,"Г.АКТАУ,13,25,Б",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.140469,43.663839,aktau,aktau_cluster_map.html
1,114.0,6127603.0,11860.0,г.Семей,"г.Семей,ул.М?ратхан Бейсембаев, д.9",1,50.402369,80.204688,"Г.АКТАУ,14,18,А",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.146722,43.650292,aktau,aktau_cluster_map.html
2,89.0,7930994.0,797.0,г.Актобе,"г.Актобе,ул.Братьев Жубановых, д.42",1,50.314685,57.132439,"Г.АКТАУ,14,26",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.146407,43.651682,aktau,aktau_cluster_map.html
3,90.0,671046.0,2.0,г.Алматы,"г.Алматы,мкр.Нуршашкан, ул. Орен, д.30",1,43.336092,77.016684,"Г.АКТАУ,14,35,*",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.144063,43.654108,aktau,aktau_cluster_map.html
4,64.0,6479898.0,3302.0,г.Уральск,"г.Уральск,ул.Алматы, д.42",1,51.230761,51.415102,"Г.АКТАУ,14,57",,МЖД с xDSL,Кластер 3,МЖД с xDSL,#3cb44b,51.144125,43.658602,aktau,aktau_cluster_map.html
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69226,90.0,275857.0,2.0,г.Алматы,"г.Алматы,ул.Сергея Вавилова, д.21",1,43.298069,76.941072,,,ЧС без xDSL,Кластер 115,ЧС без xDSL,#fabebe,76.845962,43.284414,almaty,almaty_cluster_map.html
69227,90.0,320542.0,2.0,г.Алматы,"г.Алматы,мкр.Акбулак, ул.Пригородная, д.17",1,43.247507,76.835574,,,ЧС без xDSL,Кластер 115,ЧС без xDSL,#fabebe,76.845667,43.284331,almaty,almaty_cluster_map.html
69228,90.0,282778.0,2.0,г.Алматы,"г.Алматы,ул.КОММУНАРОВ, д.18",1,43.318215,76.967932,,,ЧС без xDSL,Кластер 115,ЧС без xDSL,#fabebe,76.844839,43.284602,almaty,almaty_cluster_map.html
69229,48.0,7372287.0,5792.0,г.Караганда,"г.Караганда,ул.ДНЕПРОВСКАЯ, д.41",1,49.964448,73.229081,,,ЧС без xDSL,Кластер 115,ЧС без xDSL,#fabebe,76.845077,43.284676,almaty,almaty_cluster_map.html
